In [1]:
import os, re, hashlib, unicodedata, warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.collections as mcoll
import matplotlib.patches as mpatches

warnings.filterwarnings('ignore')
plt.rcParams['font.family'] = 'Arial'

In [ ]:
cd C:\Github_repository\Metabolomics_Tutorial\Data

In [3]:
# 0) INPUT FILES / BASIC SETTINGS
LOG_XLSX  = 'Compounds_merged_log.xlsx'     # log intensity
RAW_XLSX  = 'Compounds_merged.xlsx'         # raw intensity (대표 FeatureID 고를 때 사용)
STAT_XLSX = 'stats_merged.xlsx'             # p/q/lfc

STYPE = 'fecal_neg'                         # 예: fecal_neg / fecal_pos / serum_neg / serum_pos
WEEKS = ['0w','1w','2w']                  # serum이면 ['2w']

GROUP_COL   = 'Group'
GROUP_ORDER = ['Control', 'VNAM']

SAVE_DPI = 500
FIGSIZE  = (len(WEEKS) * 2.6 + 4.6, 8.4)

MAIN_COLORS = ['firebrick', 'royalblue'] * len(WEEKS)
SUB_COLORS  = ['maroon', 'midnightblue'] * len(WEEKS)

PLOT_MODES = [
    'bar_scatter',
    'box_scatter_transparent',
    'box_scatter',
    'violin_box',
    'swarm',
]

In [4]:
# 1) UTIL
SAMPLE_RE = re.compile(r"^[CV]\d{2}_[0-9]+w$", re.IGNORECASE)  #Edit

n_weeks = len(WEEKS)

def norm_key(s):
    return str(s).strip().lower()

def safe_filename(s, maxlen=120):
    s = str(s)
    s = unicodedata.normalize("NFKC", s)
    s = "".join(ch for ch in s if unicodedata.category(ch)[0] != "C")
    for ch in ["\u2212", "\u2010", "\u2011", "\u2012", "\u2013", "\u2014", "\u2015"]:
        s = s.replace(ch, "-")
    s = re.sub(r'[<>:"/\\|?*]+', "_", s)
    s = re.sub(r"\s+", " ", s).strip().rstrip(" .")
    if not s:
        s = "NA"
    if len(s) > maxlen:
        h = hashlib.md5(s.encode("utf-8")).hexdigest()[:8]
        s = s[:maxlen - 9].rstrip(" .") + "_" + h
    return s

def q_to_stars(q):
    if pd.isna(q):
        return 'N/A'
    if q < 0.0001:
        return '****'
    if q < 0.001:
        return '***'
    if q < 0.01:
        return '**'
    if q < 0.05:
        return '*'
    return 'ns'

def add_bracket(ax, x1, x2, y, h, text):
    ax.plot([x1, x1, x2, x2], [y, y + (0.5-0.1*(3-n_weeks))*h, y + (0.5-0.1*(3-n_weeks))*h, y], lw=1, c="k")
    text_y = y + 0.1*h
    if text in ['ns', 'N/A']:
        text_y = y + 0.7*h
    ax.text((x1+x2)/2, text_y, text, ha="center", va="bottom", fontsize=30)

def get_sample_cols(df):
    return [c for c in df.columns if SAMPLE_RE.match(str(c).strip())]

In [5]:
# 2) DISPLAY NAME 변경용
#    - 내부 매칭에는 사용 안 함
#    - figure title / 파일명 표시용
##########

DISPLAY_NAME_OVERRIDES = {
    'L-Tryptophan': 'Tryptophan',
    'DL-Tryptophan': 'Tryptophan',
#    'Melatonin',
#    'N-Acetylserotonin',
#    'Serotonin',
#    '5-Hydroxyindole-3-acetic acid',
    '5-Hydroxy-DL-tryptophan': '5-Hydroxy-tryptophan',
    'L-Kynurenine': 'Kynurenine',
#    'Kynurenine',
    '3-hydroxy-D-kynurenine': '3-hydroxy-kynurenine',
#    'Indole-3-acetaldehyde',
#    'Tryptamine',
#    'Indole-3-pyruvic acid',
    'INDOLE-3-PYRUVATE': 'Indole-3-pyruvic acid',
#    'N1-Acetyl-5-methoxykynuramine',
#    '6-Hydroxymelatonin',
#    '5-Methoxyindoleacetic acid',
#    '4-(2-Amino-3-hydroxyphenyl)-2,4-dioxobutanoic acid',
#    '4-(2-Aminophenyl)-2,4-dioxobutanoic acid',
#    'Indole-3-acetic acid',
#    '3-hydroxy-3-methyloxindole',
#    'Indole',
#    'Oxindole-3-acetic acid',
#    'Tryptophol',
#    'Kynurenic acid',
#    '6-Hydroxykynurenic acid',
    'dl-3-Indolelactic acid': 'Indole-3-lactic acid',
#    'Indole-3-lactic acid',
#    'Methyl indole-3-acetate',

#    'PC(18:3(9Z,12Z,15Z)/18:2(9Z,12Z))',
#    'PC(18:1/0:0)',
#    'LysoPC(18:3(9Z,12Z,15Z))',
#    'LysoPC(P-18:0)',
#    'LysoPC(20:5(5Z,8Z,11Z,14Z,17Z))',
    'L-α-lysophosphatidylcholine': 'LysoPC',
    'lysophosphatidylethanolamine (22:6(4Z,7Z,10Z,13Z,16Z,19Z)/0:0)': 'LysoPE(22:6(4Z,7Z,10Z,13Z,16Z,19Z)/0:0)',
    'lysophosphatidylcholine 14:1(9Z)/0:0': 'LysoPC(14:1(9Z)/0:0)',
    'Lysolecithin': 'LysoPC',
#    'Choline',
#    'Acetylcholine',
    'lysophosphatidylserine': 'LysoPS',

#    '2-Oxoglutaric acid',
#    'Succinic acid',
    'L-(-)-Malic acid': 'Malic acid',
    '(Z)-Aconitic Acid': 'Aconitic acid',
    'trans-Aconitic acid': 'Aconitic acid',
#    'Citric acid',
#    'Fumaric acid',

#    'Acetic acid',
    'Acetate': 'Acetic acid',
#    'Propionic acid',
    'Propionate': 'Propionic acid',
#    'Butyric acid',
    'Butyrate': 'Butyric acid',

#    'Proline',
    'L-Proline': 'Proline',
    'D-(+)-Proline': 'Proline',

    'L-(−)-methionine': 'Methionine',
#    'Methionine",
}


# 3) 같은 Name 중 대표 FeatureID를 강제로 지정하고 싶을 때만 사용
#    - 없으면 자동으로 raw intensity 합이 가장 큰 FeatureID 사용
##########

FORCE_REP_FEATUREID = {
    # 예시
    # ('fecal_neg', '2-Oxoglutaric acid'): '145.01435-1.439',
}

In [6]:
# 4) RAW FILE에서 대표 FeatureID 선정
#    - 같은 Name이 여러 peak로 잡히면
#      raw intensity 합이 가장 큰 FeatureID 하나만 대표로 사용
##########

def build_feature_meta(stype):
    raw = pd.read_excel(RAW_XLSX, sheet_name=stype).copy()

    raw["FeatureID"] = raw["FeatureID"].astype(str).str.strip()
    raw["Name"] = raw["Name"].fillna(raw["FeatureID"]).astype(str).str.strip()

    sample_cols = get_sample_cols(raw)
    raw[sample_cols] = raw[sample_cols].apply(pd.to_numeric, errors="coerce")
    raw["RepIntensitySum"] = raw[sample_cols].sum(axis=1, skipna=True)

    rep = (
        raw.sort_values(["Name", "RepIntensitySum", "FeatureID"], ascending=[True, False, True])
           .drop_duplicates("Name", keep="first")
           .loc[:, ["Name", "FeatureID", "RepIntensitySum", "m/z", "RT [min]"]]
           .rename(columns={
               "Name": "OriginalName",
               "FeatureID": "RepFeatureID",
               "m/z": "mz",
               "RT [min]": "RT_min"
           })
           .copy()
    )

    # 수동 강제 지정
    for (sheet_key, name_key), fid in FORCE_REP_FEATUREID.items():
        if sheet_key == stype:
            rep.loc[rep["OriginalName"] == name_key, "RepFeatureID"] = str(fid).strip()

    # 강제 지정 후 mz/RT/intensity 다시 맞춤
    raw_sub = raw[["FeatureID", "RepIntensitySum", "m/z", "RT [min]"]].copy()
    raw_sub["FeatureID"] = raw_sub["FeatureID"].astype(str).str.strip()

    rep = rep.drop(columns=["RepIntensitySum", "mz", "RT_min"])
    rep = rep.merge(raw_sub, left_on="RepFeatureID", right_on="FeatureID", how="left")
    rep = rep.drop(columns=["FeatureID"])
    rep = rep.rename(columns={"m/z": "mz", "RT [min]": "RT_min"})

    rep["DisplayName"] = rep["OriginalName"].replace(DISPLAY_NAME_OVERRIDES)

    return rep


feature_meta = build_feature_meta(STYPE)

# FeatureID -> 원래 Name / 표시 Name
featureid_to_name  = dict(zip(feature_meta["RepFeatureID"], feature_meta["OriginalName"]))
featureid_to_label = dict(zip(feature_meta["RepFeatureID"], feature_meta["DisplayName"]))

# exact Name lookup
name_to_fid = {}
for _, r in feature_meta.iterrows():
    k = norm_key(r["OriginalName"])
    if k not in name_to_fid:
        name_to_fid[k] = r["RepFeatureID"]

# DisplayName lookup (중복 없는 것만 허용)
disp_group = feature_meta.groupby(feature_meta["DisplayName"].map(norm_key))["RepFeatureID"].apply(list)
display_to_fid = {k: v[0] for k, v in disp_group.items() if len(v) == 1}

# 확인용 저장
feature_meta.to_excel(f"{STYPE}_representative_feature_map.xlsx", index=False)

In [7]:
# 5) LOG FILE 로드
#    - 컬럼명은 끝까지 FeatureID 유지
##########

mats = {}
dfs_all = []

for w in WEEKS:
    sheet_name = f"{STYPE}_{w}"

    df0 = pd.read_excel(LOG_XLSX, sheet_name=sheet_name, index_col=0)

    group = df0.loc["Group"].astype(str)
    X = df0.drop(index="Group").apply(pd.to_numeric, errors="coerce")
    X.index = X.index.astype(str).str.strip()

    mat = X.T.copy()
    mat.index = mat.index.to_series().str.replace(r"\.\d+$", "", regex=True)

    mat.insert(0, "Group", group.values)
    mat = mat.reset_index().rename(columns={"index": "SampleID"})
    mat.insert(0, "Week", w)

    mats[w] = mat
    dfs_all.append(mat)

df = pd.concat(dfs_all, ignore_index=True)

print({k: v.shape for k, v in mats.items()})
print("df:", df.shape)

{'0w': (15, 9088), '1w': (21, 9088), '2w': (21, 9088)}
df: (57, 9088)


In [8]:
# 6) KEYWORD
#    - KEYWORDS: 후보 추리기
#    - 없는 이름은 자동 skip
##########

KEYWORDS = [
    'indol', 'trypt', 'kynu', 'anthranil', 'quinoli', 'serotonin', 'melatonin',
    'coa', 'coenzyme', 'adip', 'semialdehyde', 'mucon', 'cinn', 'butano',
    'pc', 'ethanolamin', 'glycero', 'phosphatid', 'lyso', 'serin', 'cardiolipin',
    'acetaldehyde', 'lecithin', 'cholin',
    'lysine', 'glutar', 'phosph', 'hydroxypropyl', 'succin', 'citr', 'mal',
    'aconit', 'fumar',
    'acetic acid', 'acetate', 'propionic acid', 'propionate', 'butyric acid', 'butyrate',
    'proline', 'methionine',
]

EXCLUDE = []

In [9]:
# 7) KEYWORDS로 후보 추리기 (검색은 Name/DisplayName 기준)
#    - 하지만 실제 저장되는 feature_cols는 FeatureID
##########

kw = [k.lower() for k in KEYWORDS]

search_df = feature_meta.copy()
search_df["SearchText"] = (
    search_df["OriginalName"].fillna("").astype(str) + " || " +
    search_df["DisplayName"].fillna("").astype(str)
)

candidate_mask = search_df["SearchText"].str.lower().apply(
    lambda s: any(k in s for k in kw)
)
candidate_df = search_df[candidate_mask].copy()

if EXCLUDE:
    exclude_norm = {norm_key(x) for x in EXCLUDE}
    candidate_df = candidate_df[
        ~candidate_df["OriginalName"].map(norm_key).isin(exclude_norm)
    ].copy()

print("Matched candidate metabolites:", len(candidate_df))
for x in candidate_df["OriginalName"].tolist():
    print(" -", x)

Matched candidate metabolites: 289
 - (11α)-3,26-Dioxo-22,26-epoxycholest-4-en-11-yl acetate
 - (1S,2R,5R,6R,7S,8R,10S,12R,13S)-8,12-Dihydroxy-6,12-bis(hydroxymethyl)-2,6-dimethyltetracyclo[11.2.1.0~1,10~.0~2,7~]hexadec-5-yl 3-methylbutanoate
 - (1S,2R,5S)-2-Isopropyl-5-methylcyclohexyl 3-oxobutanoate
 - (1S,2R,7R,8R,11R,12R)-12-Methyl-6-methylene-17-oxa-14-azahexacyclo[10.6.3.1~5,8~.0~1,11~.0~2,8~.0~14,18~]docos-7-yl acetate
 - (1S,3R,5Z,7E)-24-(Dimethylphosphoryl)-9,10-secochola-5,7,10-triene-1,3-diol
 - (2E,6E)-farnesyl monophosphate
 - (2R)-1-(Decanoyloxy)-3-(phosphonooxy)-2-propanyl tridecanoate
 - (2R)-1-(Octanoyloxy)-3-(phosphonooxy)-2-propanyl tridecanoate
 - (2R)-1-{[(2-Aminoethoxy)(hydroxy)phosphoryl]oxy}-3-hydroxy-2-propanyl (9Z,12Z,15Z)-9,12,15-octadecatrienoate
 - (2R)-2-(Acryloyloxy)-3-(hexadecyloxy)propyl 2-(trimethylammonio)ethyl phosphate
 - (2R)-2-Hydroxy-3-(phosphonooxy)propyl (11Z)-11-icosenoate
 - (2R)-2-Hydroxy-3-(phosphonooxy)propyl (9Z)-9-nonadecenoate
 - (2R)-2

In [10]:
# 8) SELECT FEATURE
#    - SELECT_FEATURES: 최종 선택
#    - 없는 이름은 자동 skip
##########

SELECT_FEATURES = [
    'L-Tryptophan',
    'DL-Tryptophan',
    'Melatonin',
    'N-Acetylserotonin',
    'Serotonin',
    '5-Hydroxyindole-3-acetic acid',
    '5-Hydroxy-DL-tryptophan',
    'L-Kynurenine',
    'Kynurenine',
    '3-hydroxy-D-kynurenine',
    'Indole-3-acetaldehyde',
    'Tryptamine',
    'Indole-3-pyruvic acid',
    'INDOLE-3-PYRUVATE',
    'N1-Acetyl-5-methoxykynuramine',
    '6-Hydroxymelatonin',
    '5-Methoxyindoleacetic acid',
    '4-(2-Amino-3-hydroxyphenyl)-2,4-dioxobutanoic acid',
    '4-(2-Aminophenyl)-2,4-dioxobutanoic acid',
    'Indole-3-acetic acid',
    '3-hydroxy-3-methyloxindole',
    'Indole',
    'Oxindole-3-acetic acid',
    'Tryptophol',
    'Kynurenic acid',
    '6-Hydroxykynurenic acid',
    'dl-3-Indolelactic acid',
    'Indole-3-lactic acid',
    'Methyl indole-3-acetate',

    'PC(18:3(9Z,12Z,15Z)/18:2(9Z,12Z))',
    'PC(18:1/0:0)',
    'LysoPC(18:3(9Z,12Z,15Z))',
    'LysoPC(P-18:0)',
    'LysoPC(20:5(5Z,8Z,11Z,14Z,17Z))',
    'L-α-lysophosphatidylcholine',
    'lysophosphatidylethanolamine (22:6(4Z,7Z,10Z,13Z,16Z,19Z)/0:0)',
    'lysophosphatidylcholine 14:1(9Z)/0:0',
    'Lysolecithin',
    'Choline',
    'Acetylcholine',
    'lysophosphatidylserine',

    '2-Oxoglutaric acid',
    'Succinic acid',
    'L-(-)-Malic acid',
    '(Z)-Aconitic Acid',
    'trans-Aconitic acid',
    'Citric acid',
    'Fumaric acid',

    'Acetic acid',
    'Acetate',
    'Propionic acid',
    'Propionate',
    'Butyric acid',
    'Butyrate',

    'Proline',
    'L-Proline',
    'D-(+)-Proline',

    'L-(−)-methionine',
    'Methionine',
]

In [11]:
# 9) SELECT_FEATURES 적용
#    - 찾으면 FeatureID로 선택
#    - 못 찾으면 skip
##########

feature_cols = []

if len(SELECT_FEATURES) > 0:
    SELECT_FEATURES = list(dict.fromkeys(SELECT_FEATURES))

    cand_name_to_fid = {}
    for _, r in candidate_df.iterrows():
        k = norm_key(r["OriginalName"])
        if k not in cand_name_to_fid:
            cand_name_to_fid[k] = r["RepFeatureID"]

    cand_disp_groups = candidate_df.groupby(candidate_df["DisplayName"].map(norm_key))["RepFeatureID"].apply(list)
    cand_display_to_fid = {k: v[0] for k, v in cand_disp_groups.items() if len(v) == 1}

    matched = []
    missing = []

    for x in SELECT_FEATURES:
        k = norm_key(x)

        if k in cand_name_to_fid:
            matched.append(cand_name_to_fid[k])
        elif k in cand_display_to_fid:
            matched.append(cand_display_to_fid[k])
        elif k in name_to_fid:
            matched.append(name_to_fid[k])     # 후보 밖이어도 전역 lookup 한번 더 허용
        elif k in display_to_fid:
            matched.append(display_to_fid[k])
        else:
            missing.append(x)

    # 중복 제거 (입력 순서 유지)
    seen = set()
    feature_cols = []
    for fid in matched:
        if fid not in seen:
            seen.add(fid)
            feature_cols.append(fid)

    print(f"\nRequested: {len(SELECT_FEATURES)}")
    print(f"Matched  : {len(feature_cols)}")
    print(f"Missing  : {len(missing)}")

    if missing:
        print("\n[Not found]")
        for x in missing:
            print(" -", x)
else:
    feature_cols = candidate_df["RepFeatureID"].tolist()

print("\nSelected FeatureIDs:", len(feature_cols))
for fid in feature_cols:
    print(f" - {fid}   ->   {featureid_to_label.get(fid, featureid_to_name.get(fid, fid))}")


Requested: 59
Matched  : 29
Missing  : 28

[Not found]
 - L-Tryptophan
 - Melatonin
 - N-Acetylserotonin
 - Serotonin
 - L-Kynurenine
 - Kynurenine
 - Indole-3-acetaldehyde
 - Tryptamine
 - 4-(2-Amino-3-hydroxyphenyl)-2,4-dioxobutanoic acid
 - Indole
 - Oxindole-3-acetic acid
 - Tryptophol
 - 6-Hydroxykynurenic acid
 - dl-3-Indolelactic acid
 - Methyl indole-3-acetate
 - PC(18:3(9Z,12Z,15Z)/18:2(9Z,12Z))
 - PC(18:1/0:0)
 - LysoPC(18:3(9Z,12Z,15Z))
 - LysoPC(P-18:0)
 - LysoPC(20:5(5Z,8Z,11Z,14Z,17Z))
 - lysophosphatidylcholine 14:1(9Z)/0:0
 - Choline
 - Acetylcholine
 - Acetate
 - Propionate
 - Butyric acid
 - Butyrate
 - D-(+)-Proline

Selected FeatureIDs: 29
 - 203.08237-1.087   ->   Tryptophan
 - 190.05117-7.239   ->   5-Hydroxyindole-3-acetic acid
 - 219.07772-2.061   ->   5-Hydroxy-tryptophan
 - 223.07258-6.267   ->   3-hydroxy-kynurenine
 - 202.05126-17.793   ->   Indole-3-pyruvic acid
 - 202.05128-16.408   ->   Indole-3-pyruvic acid
 - 235.109-5.862   ->   N1-Acetyl-5-methoxykyn

In [12]:
# 10) stats 로드
#    - merge key = FeatureID
##########

rows = []
for wk in WEEKS:
    sub = df[df["Week"] == wk]
    for fid in feature_cols:
        if fid not in sub.columns:
            continue
        x = pd.to_numeric(sub.loc[sub[GROUP_COL] == GROUP_ORDER[0], fid], errors="coerce").dropna()
        y = pd.to_numeric(sub.loc[sub[GROUP_COL] == GROUP_ORDER[1], fid], errors="coerce").dropna()
        rows.append({"Week": wk, "feature": fid, "n": len(x) + len(y)})

stats_df = pd.DataFrame(rows)

out = []
for wk in WEEKS:
    qdf = pd.read_excel(STAT_XLSX, sheet_name=f"{STYPE}_{wk}").copy()
    qdf["FeatureID"] = qdf["FeatureID"].astype(str).str.strip()

    keep_cols = [c for c in [
        "FeatureID", "p-val", "q-val_BKY", "lfc",
        "Ctrl_n", "Vnam_n", "Ctrl_mean", "Vnam_mean"  #Edit
    ] if c in qdf.columns]

    tmp = qdf[keep_cols].copy()
    tmp["Week"] = wk
    tmp["feature"] = tmp["FeatureID"]

    if "p-val" in tmp.columns:
        tmp["p"] = pd.to_numeric(tmp["p-val"], errors="coerce")
    if "q-val_BKY" in tmp.columns:
        tmp["q"] = pd.to_numeric(tmp["q-val_BKY"], errors="coerce")
    if "lfc" in tmp.columns:
        tmp["log2FC"] = pd.to_numeric(tmp["lfc"], errors="coerce")

    out.append(tmp[["Week", "feature", "p", "q", "log2FC", "Ctrl_n", "Vnam_n", "Ctrl_mean", "Vnam_mean"]])  #Edit

pq = pd.concat(out, ignore_index=True)

stats_df = stats_df.merge(pq, on=["Week", "feature"], how="left")
stats_df = stats_df.sort_values(["Week", "q", "p"], na_position="last").reset_index(drop=True)

print("\nq missing:", stats_df["q"].isna().sum(), "/", len(stats_df))
miss = stats_df.loc[stats_df["q"].isna(), ["Week","feature"]].drop_duplicates().sort_values(["Week","feature"])
for wk in WEEKS:
    m = miss[miss["Week"] == wk]
    if not m.empty:
        print(f"\n[{wk}] ({len(m)})")
        print("\n".join(m["feature"].astype(str).tolist()))

stats_df


q missing: 9 / 87

[0w] (3)
203.08237-1.087
204.06667-9.377
204.06669-9.615

[1w] (3)
203.08237-1.087
204.06667-9.377
204.06669-9.615

[2w] (3)
203.08237-1.087
204.06667-9.377
204.06669-9.615


,Week,feature,n,p,q,log2FC,Ctrl_n,Vnam_n,Ctrl_mean,Vnam_mean
0,0w,293.11461-6.013,15,0.002797,0.024062,-3.806483,9.0,6.0,2.639971e+06,1.886833e+05
1,0w,235.109-5.862,15,0.004795,0.026680,-3.447846,9.0,6.0,3.186318e+06,2.920008e+05
2,0w,188.03554-5.821,15,0.004795,0.026680,-2.067058,9.0,6.0,1.609136e+07,3.840133e+06
3,0w,59.01388-1.062,15,0.007954,0.030809,-0.942376,9.0,6.0,2.509173e+07,1.305711e+07
4,0w,191.0199-1.113,15,0.011988,0.036715,3.220786,9.0,6.0,1.672310e+06,1.559086e+07
...,...,...,...,...,...,...,...,...,...,...
82,2w,554.34702-15.063,21,0.805324,0.222795,0.287983,11.0,10.0,3.011161e+05,3.676424e+05
83,2w,220.06169-7.766,21,0.860259,0.236382,0.518748,11.0,10.0,1.002222e+06,1.435895e+06
84,2w,203.08237-1.087,21,NaN,NaN,NaN,NaN,NaN,NaN,NaN
85,2w,204.06667-9.377,21,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
# 11) plot용 공통 객체
##########

order = []
for wk in WEEKS:
    order += [f"{wk}_{GROUP_ORDER[0]}", f"{wk}_{GROUP_ORDER[1]}"]

centers = {wk: i*2 + 0.5 for i, wk in enumerate(WEEKS)}
vlines  = [1.5 + 2*i for i in range(len(WEEKS)-1)]

x_map = {cat: i for i, cat in enumerate(order)}

palette_main = {order[i]: MAIN_COLORS[i] for i in range(len(order))}
palette_sub  = {order[i]: SUB_COLORS[i]  for i in range(len(order))}

rng = np.random.default_rng(0)


def get_feat_label(fid):
    return featureid_to_label.get(fid, featureid_to_name.get(fid, fid))

def get_plot_df(fid):
    if fid not in df.columns:
        return None

    dat = df[["Week", "Group", fid]].copy()
    dat[fid] = pd.to_numeric(dat[fid], errors="coerce")
    dat = dat.dropna(subset=[fid])
    dat = dat[dat["Group"].isin(GROUP_ORDER)]
    dat["Group6"] = dat["Week"].astype(str) + "_" + dat["Group"].astype(str)

    if dat.empty:
        return None
    return dat

def add_common_annotations(ax, dat, fid, mode):    
    y_min, y_max = np.nanmin(dat[fid]), np.nanmax(dat[fid])
    yrng = (y_max - y_min) if np.isfinite(y_max - y_min) and (y_max - y_min) > 0 else 1.0
    h = yrng * (0.06+(3-n_weeks)*0.1)

    trans = ax.get_xaxis_transform()

    for wk in WEEKS:
        ax.text(centers[wk], 1.01, wk, transform=trans, ha="center", va="bottom", fontsize=24, fontweight="bold", clip_on=False)

    y_bracket_max = y_max

    spare_map = {
        'bar_scatter': (3+(3-n_weeks)*0.8, 0.0, 2.0),
        'box_scatter_transparent': (3+(3-n_weeks)*0.8, 0.5, 2.5),
        'box_scatter': (3+(3-n_weeks)*0.8, 0.5, 2.5),
        'violin_box': (11+(3-n_weeks)*0.8, 1.0, 3.0),
        'swarm': (3.5+(3-n_weeks)*0.8, 0.5, 2.5),
    }
    bracket_h, lower_spare, upper_spare = spare_map.get(mode, (3+(3-n_weeks), 0.0, 2.0))

    for i, wk in enumerate(WEEKS):
        s = stats_df[(stats_df["Week"] == wk) & (stats_df["feature"] == fid)]
        if s.empty:
            continue

        q = float(s["q"].iloc[0]) if pd.notna(s["q"].iloc[0]) else np.nan

        if "n" in s.columns and pd.notna(s["n"].iloc[0]):
            n = int(s["n"].iloc[0])
        elif ("Ctrl_n" in s.columns) and ("Vnam_n" in s.columns):
            n = int(s["Ctrl_n"].iloc[0] + s["Vnam_n"].iloc[0])
        else:
            n = np.nan

        subw = dat[dat["Week"] == wk]
        ymax_w = np.nanmax(subw[fid]) if not subw.empty else y_max
        y = ymax_w + h*bracket_h
        x1, x2 = i*2, i*2 + 1

        if pd.isna(q):
            add_bracket(ax, x1, x2, y, h, "N/A")
        else:
            add_bracket(ax, x1, x2, y, h, q_to_stars(q))

        y_bracket_max = max(y_bracket_max, y + 0.6*h)

        ax.text(x1 - 0.45, 0.9, f"q={q:.2e}\n(n={n})", transform=trans, ha="left", va="bottom", fontsize=22, clip_on=False, fontweight="bold")

        for txt in ax.texts:
            s_txt = txt.get_text().strip()
            if s_txt in {"*", "**", "***", "****"}:
                txt.set_fontweight("bold")

    ax.set_xlim(-0.5, len(order) - 0.5)
    ax.set_ylim(y_min, y_bracket_max)
    ax.set_xlabel("")
    ax.set_ylabel("Log10 (Intensity)", fontsize=26)
    ax.set_xticks(np.arange(len(order)))
    ax.set_xticklabels(GROUP_ORDER * len(WEEKS), rotation=0)
    ax.tick_params(axis="x", labelsize=24)

    y0, y1 = ax.get_ylim()
    ax.set_yticks(np.arange(np.floor(max(0, y0)) - lower_spare, np.floor(y1) + upper_spare, 0.5))
    ax.tick_params(axis="y", labelsize=22)

    for xv in vlines:
        ax.axvline(xv, ls="--", lw=0.6, c="k", zorder=0)


def save_plot(fig, mode, fid):
    label = get_feat_label(fid)
    out_dir = {
        'bar_scatter': 'ind_bar+scatter',
        'box_scatter_transparent': 'ind_box+scatter_transparent',
        'box_scatter': 'ind_box+scatter',
        'violin_box': 'ind_violin_box',
        'swarm': 'ind_swarm',
    }[mode]

    os.makedirs(out_dir, exist_ok=True)

    # display name이 같아도 덮어쓰기 안 되게 FeatureID 같이 넣음
    out = os.path.join(out_dir, f"{safe_filename(label)}__{safe_filename(fid)}.png")
    fig.savefig(out, dpi=SAVE_DPI, bbox_inches="tight")
    plt.close(fig)

In [ ]:
# 12) BAR + SCATTER

if "bar_scatter" in PLOT_MODES:
    for fid in feature_cols:
        dat = get_plot_df(fid)
        if dat is None:
            continue

        fig, ax = plt.subplots(figsize=FIGSIZE)

        mean_df = (dat.groupby("Group6", sort=False)[fid].agg(["mean", "sem"]).reindex(order).reset_index())

        xs = np.arange(len(order))
        bar_colors = [palette_main[g] for g in mean_df["Group6"]]

        ax.bar(xs, mean_df["mean"].values, yerr=mean_df["sem"].values, color=bar_colors, edgecolor="none", capsize=9+(3-n_weeks)*5, zorder=1,)

        base_x = dat["Group6"].map(x_map).to_numpy(dtype=float)
        jitter = rng.uniform(-0.18, 0.18, size=len(dat))
        xj = base_x + jitter
        colors = dat["Group6"].map(palette_sub).to_list()

        ax.scatter(xj, dat[fid].to_numpy(dtype=float), s=150, c=colors, zorder=3, linewidths=0)

        add_common_annotations(ax, dat, fid, 'bar_scatter')

        fig.text(0.055, 1.045, get_feat_label(fid), ha="left", va="top", fontsize=26, fontweight="bold")

        plt.tight_layout()
        save_plot(fig, "bar_scatter", fid)



# 13) BOXPLOT + SCATTER (transparent)

if "box_scatter_transparent" in PLOT_MODES:
    for fid in feature_cols:
        dat = get_plot_df(fid)
        if dat is None:
            continue

        fig, ax = plt.subplots(figsize=FIGSIZE)

        sns.boxplot(data=dat, x="Group6", y=fid, order=order, showmeans=True, meanline=True, meanprops={"color": "k", "lw": 1}, palette=palette_main, width=0.75, fliersize=0, ax=ax)

        box_patches = [p for p in ax.patches if isinstance(p, mpatches.PathPatch)]
        box_patches = box_patches[:len(order)]

        lines = ax.lines
        nbox = len(box_patches)
        k = (len(lines) // nbox) if nbox else 0

        for i, p in enumerate(box_patches):
            fc = p.get_facecolor()
            p.set_facecolor((0, 0, 0, 0))
            p.set_edgecolor(fc)
            p.set_linewidth(2)

            if k:
                for ln in lines[i*k:(i+1)*k]:
                    ln.set_color(fc)
                    ln.set_linewidth(2)

        base_x = dat["Group6"].map(x_map).to_numpy(dtype=float)
        jitter = rng.uniform(-0.18, 0.18, size=len(dat))
        xj = base_x + jitter
        colors = dat["Group6"].map(palette_sub).to_list()

        ax.scatter(xj, dat[fid].to_numpy(dtype=float), s=150, c=colors, zorder=3, linewidths=0)

        add_common_annotations(ax, dat, fid, 'box_scatter_transparent')

        fig.text(0.055, 1.045, get_feat_label(fid), ha="left", va="top", fontsize=26, fontweight="bold")

        plt.tight_layout()
        save_plot(fig, "box_scatter_transparent", fid)



# 14) BOXPLOT + SCATTER

if "box_scatter" in PLOT_MODES:
    for fid in feature_cols:
        dat = get_plot_df(fid)
        if dat is None:
            continue

        fig, ax = plt.subplots(figsize=FIGSIZE)

        sns.boxplot(data=dat, x="Group6", y=fid, order=order, palette=palette_main, width=0.75, fliersize=0, ax=ax)

        for a in getattr(ax, "artists", []):
            a.set_alpha(0.9)
            a.set_zorder(1)

        base_x = dat["Group6"].map(x_map).to_numpy(dtype=float)
        jitter = rng.uniform(-0.18, 0.18, size=len(dat))
        xj = base_x + jitter
        colors = dat["Group6"].map(palette_sub).to_list()

        ax.scatter(xj, dat[fid].to_numpy(dtype=float), s=150, c=colors, zorder=3, linewidths=0)

        add_common_annotations(ax, dat, fid, 'box_scatter')

        fig.text(0.055, 1.045, get_feat_label(fid), ha="left", va="top", fontsize=26, fontweight="bold")

        plt.tight_layout()
        save_plot(fig, "box_scatter", fid)



# 15) VIOLIN + BOX

if "violin_box" in PLOT_MODES:
    for fid in feature_cols:
        dat = get_plot_df(fid)
        if dat is None:
            continue

        fig, ax = plt.subplots(figsize=FIGSIZE)

        before = len(ax.collections)

        sns.violinplot(data=dat, x="Group6", y=fid, order=order, palette=palette_main, inner="box", ax=ax, inner_kws=dict(box_width=10, whis_width=1, color="#444444"))

        sns.boxplot(data=dat, x="Group6", y=fid, order=order, width=0.4, showmeans=True, showbox=False, medianprops={"visible": False}, whiskerprops={"visible": False}, showcaps=False, meanline=True, showfliers=False, meanprops={"color": "k", "lw": 1}, ax=ax)

        for c in ax.collections:
            c.set_alpha(0.9)
            c.set_zorder(2)

        violin_cols = [c for c in ax.collections[before:] if isinstance(c, mcoll.PolyCollection)]
        for pc in violin_cols:
            pc.set_alpha(0.9)
            pc.set_zorder(1)

        add_common_annotations(ax, dat, fid, 'violin_box')

        fig.text(0.055, 1.045, get_feat_label(fid), ha="left", va="top", fontsize=26, fontweight="bold")

        plt.tight_layout()
        save_plot(fig, "violin_box", fid)



# 16) SWARMPLOT

if "swarm" in PLOT_MODES:
    for fid in feature_cols:
        dat = get_plot_df(fid)
        if dat is None:
            continue

        fig, ax = plt.subplots(figsize=FIGSIZE)

        sns.swarmplot(data=dat, x="Group6", y=fid, order=order, palette=palette_sub, size=15, ax=ax)

        sns.boxplot(data=dat, x="Group6", y=fid, order=order, width=0.6, showmeans=True, showbox=False, medianprops={"visible": False}, whiskerprops={"visible": False}, showcaps=False, meanline=True, showfliers=False, meanprops={"color": "k", "lw": 1}, ax=ax)

        for c in ax.collections:
            c.set_alpha(0.9)
            c.set_zorder(2)

        add_common_annotations(ax, dat, fid, 'swarm')

        fig.text(0.055, 1.045, get_feat_label(fid), ha="left", va="top", fontsize=26, fontweight="bold")

        plt.tight_layout()
        save_plot(fig, "swarm", fid)